## chargement de Yellow nettoyé

On ouvre le fichier `clean_data/yellow_clean.csv.gz` produit par le notebook de nettoyage. Pandas décompresse le fichier tout seul, et on redonne à chaque colonne son type (zones en entiers, mesures en nombres, dates en vraies dates). On crée aussi le dossier `agg_data` qui recevra les résumés par zone.

**Effet sur les données :** les courses Yellow nettoyées sont en mémoire, une ligne par course.

In [1]:
import os, time, gc, warnings
import numpy as np
import pandas as pd
from IPython.display import display
warnings.filterwarnings("ignore")

CLEAN_DIR = os.path.join(os.getcwd(), "clean_data")
AGG_DIR = os.path.join(os.getcwd(), "agg_data")
os.makedirs(AGG_DIR, exist_ok=True)

TYPES = {"zone_depart": "int16", "zone_arrivee": "float32", "distance": "float32", "prix": "float32",
         "pourboire": "float32", "paiement": "float32", "frais_cbd": "float32", "duree_min": "float32"}

t0 = time.time()
yellow = pd.read_csv(os.path.join(CLEAN_DIR, "yellow_clean.csv.gz"), dtype=TYPES, parse_dates=["date_debut", "date_fin"])
print("Lignes :", len(yellow), "| colonnes :", yellow.shape[1])
print("Temps (s) :", round(time.time() - t0))
display(yellow.head())

Lignes : 14269897 | colonnes : 10
Temps (s) : 67


,date_debut,date_fin,zone_depart,zone_arrivee,distance,prix,pourboire,paiement,frais_cbd,duree_min
0,2024-02-01 00:04:45,2024-02-01 00:19:58,68,236.0,4.390000,20.5,1.28,1.0,0.0,15.216666
1,2024-02-01 00:56:31,2024-02-01 01:10:53,48,243.0,7.710000,31.0,9.00,1.0,0.0,14.366667
2,2024-02-01 00:07:50,2024-02-01 00:43:12,132,261.0,28.690001,70.0,0.00,2.0,0.0,35.366665
3,2024-02-01 00:01:49,2024-02-01 00:10:47,161,163.0,1.100000,9.3,2.85,1.0,0.0,8.966666
4,2024-02-01 00:37:35,2024-02-01 00:51:15,246,79.0,2.600000,15.6,0.00,2.0,0.0,13.666667


## préparation des compteurs et des sommes

Pour chaque course, on crée des colonnes de **comptage** (1 si la course remplit une condition, 0 sinon) et des colonnes de **somme** (la valeur de la mesure si elle est valide, 0 sinon). Une fois additionnées par zone à la cellule suivante, elles donneront les bons totaux.

**Comptages (`n_`)**
- `n` : la course elle-même (1 pour chaque course)
- `n_2024`, `n_2025` : courses de chaque année, pour la croissance
- `n_weekend` : courses du samedi et du dimanche
- `n_nuit` (0-6 h), `n_matin` (6-10 h), `n_journee` (10-16 h), `n_fin_aprem` (16-20 h), `n_soiree` (20-24 h) : les 5 tranches horaires
- `n_aeroport` : courses qui arrivent à un aéroport (zones 1 Newark, 132 JFK, 138 LaGuardia)
- `n_duree`, `n_dist`, `n_prix` : nombre de mesures valides, qui serviront de dénominateurs pour les moyennes
- `n_paiement`, `n_especes` : paiements connus (carte ou espèces) et paiements en espèces
- `n_cbd` : courses de 2025 qui ont payé le péage CBD

**Sommes (`s_`)**
- `s_duree`, `s_dist`, `s_prix` : sommes des durées, distances et prix valides
- `s_prix_pm`, `s_dist_pm` : prix et distance des courses où les deux sont valides, pour le prix par mile
- `s_pourboire`, `s_prix_pb` : pourboire et prix des paiements par carte, seuls à avoir un pourboire enregistré, pour le taux de pourboire

On garde des **sommes** et non des moyennes parce que les moyennes ne s'additionnent pas entre services : avec les sommes, on pourra combiner Yellow, Green et les VTC correctement à l'étape d'intégration.

**Effet sur les données :** une table de travail avec une ligne par course et 23 colonnes de comptage et de somme.

In [2]:
AEROPORTS = [1, 132, 138]
heure = yellow["date_debut"].dt.hour
annee = yellow["date_debut"].dt.year

ind = pd.DataFrame({"zone_id": yellow["zone_depart"]})
ind["n"] = np.ones(len(yellow), dtype="int32")
ind["n_2024"] = (annee == 2024).astype("int32")
ind["n_2025"] = (annee == 2025).astype("int32")
ind["n_weekend"] = (yellow["date_debut"].dt.dayofweek >= 5).astype("int32")
ind["n_nuit"] = (heure < 6).astype("int32")
ind["n_matin"] = ((heure >= 6) & (heure < 10)).astype("int32")
ind["n_journee"] = ((heure >= 10) & (heure < 16)).astype("int32")
ind["n_fin_aprem"] = ((heure >= 16) & (heure < 20)).astype("int32")
ind["n_soiree"] = (heure >= 20).astype("int32")
ind["n_aeroport"] = yellow["zone_arrivee"].isin(AEROPORTS).astype("int32")

ind["n_duree"] = yellow["duree_min"].notna().astype("int32")
ind["s_duree"] = yellow["duree_min"].fillna(0).astype("float64")
ind["n_dist"] = yellow["distance"].notna().astype("int32")
ind["s_dist"] = yellow["distance"].fillna(0).astype("float64")
ind["n_prix"] = yellow["prix"].notna().astype("int32")
ind["s_prix"] = yellow["prix"].fillna(0).astype("float64")

prix_et_dist = yellow["prix"].notna() & yellow["distance"].notna()
ind["s_prix_pm"] = yellow["prix"].where(prix_et_dist, 0).astype("float64")
ind["s_dist_pm"] = yellow["distance"].where(prix_et_dist, 0).astype("float64")

carte_ok = (yellow["paiement"] == 1) & yellow["prix"].notna() & yellow["pourboire"].notna()
ind["s_pourboire"] = yellow["pourboire"].where(carte_ok, 0).astype("float64")
ind["s_prix_pb"] = yellow["prix"].where(carte_ok, 0).astype("float64")

ind["n_paiement"] = yellow["paiement"].isin([1, 2]).astype("int32")
ind["n_especes"] = (yellow["paiement"] == 2).astype("int32")
ind["n_cbd"] = ((annee == 2025) & (yellow["frais_cbd"] > 0)).astype("int32")

del heure, annee, prix_et_dist, carte_ok
gc.collect()
print("Colonnes préparées :", ind.shape[1] - 1)
display(ind.head())

Colonnes préparées : 23


,zone_id,n,n_2024,n_2025,n_weekend,n_nuit,n_matin,n_journee,n_fin_aprem,n_soiree,...,s_dist,n_prix,s_prix,s_prix_pm,s_dist_pm,s_pourboire,s_prix_pb,n_paiement,n_especes,n_cbd
0,68,1,1,0,0,1,0,0,0,0,...,4.390000,1,20.5,20.5,4.390000,1.28,20.5,1,0,0
1,48,1,1,0,0,1,0,0,0,0,...,7.710000,1,31.0,31.0,7.710000,9.00,31.0,1,0,0
2,132,1,1,0,0,1,0,0,0,0,...,28.690001,1,70.0,70.0,28.690001,0.00,0.0,1,1,0
3,161,1,1,0,0,1,0,0,0,0,...,1.100000,1,9.3,9.3,1.100000,2.85,9.3,1,0,0
4,246,1,1,0,0,1,0,0,0,0,...,2.600000,1,15.6,15.6,2.600000,0.00,0.0,1,1,0


In [3]:
display(ind.columns)



Index(['zone_id', 'n', 'n_2024', 'n_2025', 'n_weekend', 'n_nuit', 'n_matin',
       'n_journee', 'n_fin_aprem', 'n_soiree', 'n_aeroport', 'n_duree',
       's_duree', 'n_dist', 's_dist', 'n_prix', 's_prix', 's_prix_pm',
       's_dist_pm', 's_pourboire', 's_prix_pb', 'n_paiement', 'n_especes',
       'n_cbd'],
      dtype='object')

## passage de la course à la zone

On regroupe toutes les courses par zone de départ et on additionne chaque colonne. Les 14 millions de lignes deviennent une ligne par zone. Chaque colonne reçoit le préfixe `yellow_`, pour qu'on sache d'où elle vient quand on joindra Green, FHV et HVFHV. Les sommes sont arrondies à 2 décimales.

**Effet sur les données :** on passe d'environ 14 millions de courses à environ 260 zones et 24 colonnes (le numéro de zone et 23 totaux). On affiche les 10 zones les plus actives.

In [4]:
yellow_zone = ind.groupby("zone_id").sum()
yellow_zone[[c for c in yellow_zone.columns if c.startswith("s_")]] = yellow_zone[[c for c in yellow_zone.columns if c.startswith("s_")]].round(2)
yellow_zone.columns = ["yellow_" + c for c in yellow_zone.columns]
yellow_zone = yellow_zone.reset_index()

del ind
gc.collect()
print("Zones :", len(yellow_zone), "| colonnes :", yellow_zone.shape[1])
display(yellow_zone.sort_values("yellow_n", ascending=False).head(10))

Zones : 259 | colonnes : 24


,zone_id,yellow_n,yellow_n_2024,yellow_n_2025,yellow_n_weekend,yellow_n_nuit,yellow_n_matin,yellow_n_journee,yellow_n_fin_aprem,yellow_n_soiree,...,yellow_s_dist,yellow_n_prix,yellow_s_prix,yellow_s_prix_pm,yellow_s_dist_pm,yellow_s_pourboire,yellow_s_prix_pb,yellow_n_paiement,yellow_n_especes,yellow_n_cbd
156,161,653929,310347,343582,131307,17611,46835,212064,221710,155709,...,1497963.62,636846,10166574.25,9969753.25,1463349.80,1957673.99,8023988.08,579178,76762,336908
232,237,624666,296425,328241,144812,12790,73848,247325,182366,108337,...,1104192.63,611844,7889094.09,7769167.91,1082626.33,1567073.00,6245531.69,569107,76828,171889
231,236,581133,280037,301096,136787,6971,101782,242655,164220,65505,...,1124660.32,568327,7654102.54,7509761.67,1099521.10,1450560.19,5930569.76,516192,64687,98692
127,132,571834,284479,287355,178540,50700,65381,148845,157261,149647,...,8780651.18,544201,34312777.51,33876673.50,8470474.48,5093443.20,26856836.29,525107,124726,110911
225,230,476627,225240,251387,137994,34594,38613,122958,128885,151577,...,1381992.78,459637,8474692.14,8254124.59,1341231.13,1503684.35,6338102.34,413692,71806,244092
157,162,472360,229731,242629,97331,14350,58132,149516,155726,94636,...,1057850.31,461557,7158768.01,7017171.04,1037077.72,1405481.79,5777908.18,427463,53758,237875
181,186,461741,215177,246564,134320,30626,78137,149702,108030,95246,...,1028382.14,450334,7367755.70,7249914.48,1005003.83,1378229.94,5845860.82,421548,61896,240030
137,142,434016,212224,221792,130669,13226,47661,136071,116338,120720,...,923585.92,422474,5973892.44,5859480.20,899960.53,1138164.21,4578918.31,382581,51930,131002
229,234,398695,181752,216943,117051,29755,27494,112938,117283,111225,...,800431.48,386925,5593831.65,5464704.94,777941.98,1016119.55,4155154.54,331396,36250,213635
165,170,397325,189468,207857,98329,19244,47429,133524,114574,82554,...,874695.88,385376,5939398.47,5775167.14,850495.85,1061163.32,4448858.69,336647,44772,203194


## contrôle et sauvegarde

On vérifie que l'agrégation n'a ni perdu ni dupliqué de courses :
- la somme de `yellow_n` sur toutes les zones doit être égale au nombre de lignes de la table nettoyée ;
- courses 2024 + courses 2025 = total ;
- la somme des 5 tranches horaires = total.

Puis on sauvegarde le résumé dans `agg_data/yellow_zone.csv`. Le fichier est très petit, une ligne par zone, donc il n'a pas besoin d'être compressé.

**Effet sur les données :** aucun changement, le résumé Yellow par zone est gardé sur le disque pour l'intégration.

In [5]:
print("Courses dans la table nettoyée :", len(yellow))
print("Courses dans le résumé par zone :", yellow_zone["yellow_n"].sum())
print("2024 + 2025 = total :", (yellow_zone["yellow_n_2024"] + yellow_zone["yellow_n_2025"]).sum() == yellow_zone["yellow_n"].sum())
tranches = ["yellow_n_nuit", "yellow_n_matin", "yellow_n_journee", "yellow_n_fin_aprem", "yellow_n_soiree"]
print("Somme des 5 tranches = total :", yellow_zone[tranches].sum(axis=1).sum() == yellow_zone["yellow_n"].sum())

cible = os.path.join(AGG_DIR, "yellow_zone.csv")
yellow_zone.to_csv(cible, index=False)
print("Sauvegardé :", cible)

Courses dans la table nettoyée : 14269897
Courses dans le résumé par zone : 14269897
2024 + 2025 = total : True
Somme des 5 tranches = total : True
Sauvegardé : c:\Users\DELL\Desktop\esprit\ML\notebook_v0\Nootebook_vf\agg_data\yellow_zone.csv


## chargement de Green nettoyé

On libère d'abord la mémoire occupée par la table Yellow : son résumé par zone est déjà sauvegardé dans `agg_data/yellow_zone.csv`, on n'a plus besoin des 14 millions de courses. Puis on ouvre `clean_data/green_clean.csv.gz` avec les mêmes types que pour Yellow.

Green a les mêmes colonnes que Yellow (dates, zones, distance, prix, pourboire, paiement, péage CBD, durée), donc tout le reste de l'agrégation se fait de la même façon.

**Effet sur les données :** les courses Green nettoyées sont en mémoire, une ligne par course.

In [2]:
import os, time, gc, warnings
import numpy as np
import pandas as pd
from IPython.display import display
warnings.filterwarnings("ignore")

CLEAN_DIR = os.path.join(os.getcwd(), "clean_data")
AGG_DIR = os.path.join(os.getcwd(), "agg_data")
os.makedirs(AGG_DIR, exist_ok=True)

if "yellow" in globals():
    del yellow
    gc.collect()

TYPES = {"zone_depart": "int16", "zone_arrivee": "float32", "distance": "float32", "prix": "float32",
         "pourboire": "float32", "paiement": "float32", "frais_cbd": "float32", "duree_min": "float32"}

t0 = time.time()
green = pd.read_csv(os.path.join(CLEAN_DIR, "green_clean.csv.gz"), dtype=TYPES, parse_dates=["date_debut", "date_fin"])
print("Lignes :", len(green), "| colonnes :", green.shape[1])
print("Temps (s) :", round(time.time() - t0))
display(green.head())

Lignes : 208490 | colonnes : 10
Temps (s) : 1


,date_debut,date_fin,zone_depart,zone_arrivee,distance,prix,pourboire,paiement,frais_cbd,duree_min
0,2024-02-01 00:15:53,2024-02-01 00:24:20,75,161.0,2.77,13.5,3.75,1.0,0.0,8.450000
1,2024-02-01 00:30:29,2024-02-01 00:35:32,75,238.0,1.03,7.9,2.60,1.0,0.0,5.050000
2,2024-02-01 00:31:14,2024-02-01 00:31:16,95,NaN,NaN,50.0,1.00,1.0,0.0,NaN
3,2024-02-01 00:06:23,2024-02-01 00:10:10,97,25.0,0.34,5.1,0.00,2.0,0.0,3.783333
4,2024-02-01 00:30:22,2024-02-01 00:33:43,97,97.0,0.34,5.1,0.00,2.0,0.0,3.350000


## préparation des compteurs et des sommes

Mêmes 23 colonnes que pour Yellow :

- **comptages** : `n`, `n_2024`, `n_2025`, `n_weekend`, les 5 tranches horaires (`n_nuit`, `n_matin`, `n_journee`, `n_fin_aprem`, `n_soiree`), `n_aeroport`, `n_duree`, `n_dist`, `n_prix`, `n_paiement`, `n_especes`, `n_cbd` ;
- **sommes** : `s_duree`, `s_dist`, `s_prix`, `s_prix_pm`, `s_dist_pm`, `s_pourboire`, `s_prix_pb`.

Chaque course vaut 1 ou 0 dans les comptages, et la valeur de la mesure (ou 0 si elle a été vidée au nettoyage) dans les sommes. Le prix par mile n'utilise que les courses dont le prix **et** la distance sont valides, et le taux de pourboire seulement les paiements par carte, puisque les pourboires en espèces ne sont jamais enregistrés.

**Effet sur les données :** une table de travail avec une ligne par course et 23 colonnes de comptage et de somme.

In [3]:
AEROPORTS = [1, 132, 138]
heure = green["date_debut"].dt.hour
annee = green["date_debut"].dt.year

ind = pd.DataFrame({"zone_id": green["zone_depart"]})
ind["n"] = np.ones(len(green), dtype="int32")
ind["n_2024"] = (annee == 2024).astype("int32")
ind["n_2025"] = (annee == 2025).astype("int32")
ind["n_weekend"] = (green["date_debut"].dt.dayofweek >= 5).astype("int32")
ind["n_nuit"] = (heure < 6).astype("int32")
ind["n_matin"] = ((heure >= 6) & (heure < 10)).astype("int32")
ind["n_journee"] = ((heure >= 10) & (heure < 16)).astype("int32")
ind["n_fin_aprem"] = ((heure >= 16) & (heure < 20)).astype("int32")
ind["n_soiree"] = (heure >= 20).astype("int32")
ind["n_aeroport"] = green["zone_arrivee"].isin(AEROPORTS).astype("int32")

ind["n_duree"] = green["duree_min"].notna().astype("int32")
ind["s_duree"] = green["duree_min"].fillna(0).astype("float64")
ind["n_dist"] = green["distance"].notna().astype("int32")
ind["s_dist"] = green["distance"].fillna(0).astype("float64")
ind["n_prix"] = green["prix"].notna().astype("int32")
ind["s_prix"] = green["prix"].fillna(0).astype("float64")

prix_et_dist = green["prix"].notna() & green["distance"].notna()
ind["s_prix_pm"] = green["prix"].where(prix_et_dist, 0).astype("float64")
ind["s_dist_pm"] = green["distance"].where(prix_et_dist, 0).astype("float64")

carte_ok = (green["paiement"] == 1) & green["prix"].notna() & green["pourboire"].notna()
ind["s_pourboire"] = green["pourboire"].where(carte_ok, 0).astype("float64")
ind["s_prix_pb"] = green["prix"].where(carte_ok, 0).astype("float64")

ind["n_paiement"] = green["paiement"].isin([1, 2]).astype("int32")
ind["n_especes"] = (green["paiement"] == 2).astype("int32")
ind["n_cbd"] = ((annee == 2025) & (green["frais_cbd"] > 0)).astype("int32")

del heure, annee, prix_et_dist, carte_ok
gc.collect()
print("Colonnes préparées :", ind.shape[1] - 1)
display(ind.head())

Colonnes préparées : 23


,zone_id,n,n_2024,n_2025,n_weekend,n_nuit,n_matin,n_journee,n_fin_aprem,n_soiree,...,s_dist,n_prix,s_prix,s_prix_pm,s_dist_pm,s_pourboire,s_prix_pb,n_paiement,n_especes,n_cbd
0,75,1,1,0,0,1,0,0,0,0,...,2.77,1,13.5,13.5,2.77,3.75,13.5,1,0,0
1,75,1,1,0,0,1,0,0,0,0,...,1.03,1,7.9,7.9,1.03,2.60,7.9,1,0,0
2,95,1,1,0,0,1,0,0,0,0,...,0.00,1,50.0,0.0,0.00,1.00,50.0,1,0,0
3,97,1,1,0,0,1,0,0,0,0,...,0.34,1,5.1,5.1,0.34,0.00,0.0,1,1,0
4,97,1,1,0,0,1,0,0,0,0,...,0.34,1,5.1,5.1,0.34,0.00,0.0,1,1,0


## passage de la course à la zone

On regroupe les courses par zone de départ et on additionne chaque colonne. Toutes les colonnes reçoivent le préfixe `green_`, pour les distinguer de celles de Yellow lors de l'intégration. Les sommes sont arrondies à 2 décimales.

**Effet sur les données :** on passe d'environ 209 000 courses à une ligne par zone et 24 colonnes. Les taxis verts n'ayant pas le droit de prendre des clients dans la rue au centre de Manhattan, les zones les plus actives devraient se trouver au nord de Manhattan (Harlem, East Harlem), dans le Bronx, à Brooklyn et dans le Queens, et non à Midtown comme pour Yellow.

In [4]:
green_zone = ind.groupby("zone_id").sum()
green_zone[[c for c in green_zone.columns if c.startswith("s_")]] = green_zone[[c for c in green_zone.columns if c.startswith("s_")]].round(2)
green_zone.columns = ["green_" + c for c in green_zone.columns]
green_zone = green_zone.reset_index()

del ind
gc.collect()
print("Zones :", len(green_zone), "| colonnes :", green_zone.shape[1])
display(green_zone.sort_values("green_n", ascending=False).head(10))

Zones : 240 | colonnes : 24


,zone_id,green_n,green_n_2024,green_n_2025,green_n_weekend,green_n_nuit,green_n_matin,green_n_journee,green_n_fin_aprem,green_n_soiree,...,green_s_dist,green_n_prix,green_s_prix,green_s_prix_pm,green_s_dist_pm,green_s_pourboire,green_s_prix_pb,green_n_paiement,green_n_especes,green_n_cbd
66,74,46916,23792,23124,12238,899,14316,15608,11292,4801,...,105391.37,46806,684229.15,674498.62,105350.11,119547.91,550676.58,45954,8791,438
67,75,30952,15517,15435,4211,572,3062,12113,11462,3743,...,69782.23,30869,446287.64,436923.20,69749.06,73721.36,328663.16,30011,7024,2246
146,166,10771,5834,4937,2452,145,1449,4453,3102,1622,...,27226.77,10748,168633.71,166113.51,27220.58,28537.95,123081.25,10051,2282,706
86,95,10564,5925,4639,2328,586,1072,3522,3594,1790,...,26533.10,10536,168438.40,162189.77,26529.13,19173.88,92946.23,10391,4411,112
36,43,10009,5214,4795,959,26,1077,4149,3931,826,...,22660.99,9999,148771.26,146841.26,22657.09,29387.81,123841.00,9917,1533,1073
34,41,9313,5268,4045,2830,209,1502,4358,2344,900,...,20305.91,9258,131591.62,127079.42,20289.81,13703.50,74401.89,8488,3295,342
74,82,8935,5357,3578,2460,966,275,2681,3497,1516,...,25421.50,8878,156639.41,149859.92,25414.05,16564.87,85819.57,8690,3979,61
88,97,7529,3914,3615,2545,87,150,3021,2815,1456,...,18436.88,7522,130911.93,127496.74,18435.88,19949.81,101086.59,7361,1476,238
57,65,6342,3613,2729,2221,211,351,2054,2632,1094,...,18315.61,6329,121156.93,117898.50,18303.20,19604.70,95735.19,6156,1197,293
112,130,5435,2497,2938,1446,385,791,1521,1530,1208,...,22380.42,5421,120182.32,115508.92,22371.35,14165.74,90582.30,5355,1272,89


## contrôle et sauvegarde

On vérifie que l'agrégation n'a ni perdu ni dupliqué de courses :
- la somme de `green_n` sur toutes les zones = nombre de lignes de la table nettoyée ;
- courses 2024 + courses 2025 = total ;
- somme des 5 tranches horaires = total.

Puis on sauvegarde le résumé dans `agg_data/green_zone.csv`.

**Effet sur les données :** aucun changement, le résumé Green par zone est gardé sur le disque pour l'intégration.

In [5]:
print("Courses dans la table nettoyée :", len(green))
print("Courses dans le résumé par zone :", green_zone["green_n"].sum())
print("2024 + 2025 = total :", (green_zone["green_n_2024"] + green_zone["green_n_2025"]).sum() == green_zone["green_n"].sum())
tranches = ["green_n_nuit", "green_n_matin", "green_n_journee", "green_n_fin_aprem", "green_n_soiree"]
print("Somme des 5 tranches = total :", green_zone[tranches].sum(axis=1).sum() == green_zone["green_n"].sum())

cible = os.path.join(AGG_DIR, "green_zone.csv")
green_zone.to_csv(cible, index=False)
print("Sauvegardé :", cible)

Courses dans la table nettoyée : 208490
Courses dans le résumé par zone : 208490
2024 + 2025 = total : True
Somme des 5 tranches = total : True
Sauvegardé : c:\Users\DELL\Desktop\esprit\ML\notebook_v0\Nootebook_vf\agg_data\green_zone.csv


## chargement de FHV nettoyé

On refait les imports et on redéfinit les dossiers pour que la cellule fonctionne seule. On libère la mémoire de Green (son résumé est déjà dans `agg_data/green_zone.csv`), puis on ouvre `clean_data/fhv_clean.csv.gz`.

FHV n'a que 5 colonnes : `date_debut`, `date_fin`, `zone_depart`, `zone_arrivee` et `duree_min`. Les bases de VTC classiques ne déclarent ni prix, ni distance, ni paiement à la TLC.

**Effet sur les données :** les courses FHV nettoyées sont en mémoire, une ligne par course.

In [6]:
import os, time, gc, warnings
import numpy as np
import pandas as pd
from IPython.display import display
warnings.filterwarnings("ignore")

CLEAN_DIR = os.path.join(os.getcwd(), "clean_data")
AGG_DIR = os.path.join(os.getcwd(), "agg_data")
os.makedirs(AGG_DIR, exist_ok=True)

if "green" in globals():
    del green
    gc.collect()

TYPES = {"zone_depart": "int16", "zone_arrivee": "float32", "duree_min": "float32"}

t0 = time.time()
fhv = pd.read_csv(os.path.join(CLEAN_DIR, "fhv_clean.csv.gz"), dtype=TYPES, parse_dates=["date_debut", "date_fin"])
print("Lignes :", len(fhv), "| colonnes :", fhv.shape[1])
print("Temps (s) :", round(time.time() - t0))
display(fhv.head())

Lignes : 1044984 | colonnes : 5
Temps (s) : 6


,date_debut,date_fin,zone_depart,zone_arrivee,duree_min
0,2024-02-01 00:01:59,2024-02-01 00:30:41,170,NaN,28.700001
1,2024-02-01 00:15:49,2024-02-01 00:35:56,116,87.0,20.116667
2,2024-02-01 00:30:08,2024-02-01 00:44:32,87,75.0,14.400000
3,2024-02-01 00:10:26,2024-02-01 00:36:49,13,135.0,26.383333
4,2024-02-01 00:19:30,2024-02-01 00:41:21,87,251.0,21.850000


## préparation des compteurs et des sommes

FHV ne permet de calculer que 12 colonnes, contre 23 pour les taxis :

- **comptages** : `n` (la course), `n_2024`, `n_2025`, `n_weekend`, les 5 tranches horaires (`n_nuit`, `n_matin`, `n_journee`, `n_fin_aprem`, `n_soiree`), `n_aeroport` (arrivée à Newark, JFK ou LaGuardia) et `n_duree` (durées valides) ;
- **somme** : `s_duree`, total des durées valides en minutes.

La méthode est identique à celle des taxis : 1 ou 0 pour les comptages, la valeur de la durée (ou 0 si elle a été vidée) pour la somme.

**Effet sur les données :** une table de travail avec une ligne par course et 12 colonnes.

In [7]:
AEROPORTS = [1, 132, 138]
heure = fhv["date_debut"].dt.hour
annee = fhv["date_debut"].dt.year

ind = pd.DataFrame({"zone_id": fhv["zone_depart"]})
ind["n"] = np.ones(len(fhv), dtype="int32")
ind["n_2024"] = (annee == 2024).astype("int32")
ind["n_2025"] = (annee == 2025).astype("int32")
ind["n_weekend"] = (fhv["date_debut"].dt.dayofweek >= 5).astype("int32")
ind["n_nuit"] = (heure < 6).astype("int32")
ind["n_matin"] = ((heure >= 6) & (heure < 10)).astype("int32")
ind["n_journee"] = ((heure >= 10) & (heure < 16)).astype("int32")
ind["n_fin_aprem"] = ((heure >= 16) & (heure < 20)).astype("int32")
ind["n_soiree"] = (heure >= 20).astype("int32")
ind["n_aeroport"] = fhv["zone_arrivee"].isin(AEROPORTS).astype("int32")

ind["n_duree"] = fhv["duree_min"].notna().astype("int32")
ind["s_duree"] = fhv["duree_min"].fillna(0).astype("float64")

del heure, annee
gc.collect()
print("Colonnes préparées :", ind.shape[1] - 1)
display(ind.head())

Colonnes préparées : 12


,zone_id,n,n_2024,n_2025,n_weekend,n_nuit,n_matin,n_journee,n_fin_aprem,n_soiree,n_aeroport,n_duree,s_duree
0,170,1,1,0,0,1,0,0,0,0,0,1,28.700001
1,116,1,1,0,0,1,0,0,0,0,0,1,20.116667
2,87,1,1,0,0,1,0,0,0,0,0,1,14.400000
3,13,1,1,0,0,1,0,0,0,0,0,1,26.383333
4,87,1,1,0,0,1,0,0,0,0,0,1,21.850000


## passage de la course à la zone

On regroupe les courses par zone de départ et on additionne chaque colonne. Les colonnes reçoivent le préfixe `fhv_`. La somme des durées est arrondie à 2 décimales.

**Effet sur les données :** on passe de quelques millions de courses à une ligne par zone et 13 colonnes (le numéro de zone et 12 totaux). Comme le nettoyage a retiré toutes les courses sans zone de départ, il ne reste que les courses des bases qui déclarent leurs zones.

In [8]:
fhv_zone = ind.groupby("zone_id").sum()
fhv_zone["s_duree"] = fhv_zone["s_duree"].round(2)
fhv_zone.columns = ["fhv_" + c for c in fhv_zone.columns]
fhv_zone = fhv_zone.reset_index()

del ind
gc.collect()
print("Zones :", len(fhv_zone), "| colonnes :", fhv_zone.shape[1])
display(fhv_zone.sort_values("fhv_n", ascending=False).head(10))

Zones : 260 | colonnes : 13


,zone_id,fhv_n,fhv_n_2024,fhv_n_2025,fhv_n_weekend,fhv_n_nuit,fhv_n_matin,fhv_n_journee,fhv_n_fin_aprem,fhv_n_soiree,fhv_n_aeroport,fhv_n_duree,fhv_s_duree
202,206,31136,15408,15728,6320,1927,9899,13234,0,2183,75,30111,629805.13
217,221,30397,14911,15486,6189,1788,8235,13503,0,2643,61,30169,709492.42
128,132,19823,8485,11338,6178,1119,3416,6016,0,3869,543,19559,974114.38
210,214,17248,8214,9034,1952,440,3291,10014,0,713,46,17122,517718.12
55,56,17129,13456,3673,5314,1904,4742,5524,0,2314,98,17083,298804.45
119,123,16319,8363,7956,2422,661,2383,8946,0,1330,55,16257,347867.72
22,23,16079,7657,8422,2139,625,3674,8881,0,568,90,16017,410489.77
125,129,15972,9046,6926,3938,1459,3840,5946,0,2225,110,15902,304306.53
91,92,15333,9139,6194,3616,2668,2361,5765,0,2458,527,12249,258896.93
28,29,14649,7993,6656,2130,442,2641,8074,0,877,53,14560,319308.98


## contrôle et sauvegarde

On vérifie que l'agrégation n'a ni perdu ni dupliqué de courses :
- la somme de `fhv_n` sur toutes les zones = nombre de lignes de la table nettoyée ;
- courses 2024 + courses 2025 = total ;
- somme des 5 tranches horaires = total.

Puis on sauvegarde le résumé dans `agg_data/fhv_zone.csv`.

**Effet sur les données :** aucun changement, le résumé FHV par zone est gardé sur le disque pour l'intégration.

In [9]:
print("Courses dans la table nettoyée :", len(fhv))
print("Courses dans le résumé par zone :", fhv_zone["fhv_n"].sum())
print("2024 + 2025 = total :", (fhv_zone["fhv_n_2024"] + fhv_zone["fhv_n_2025"]).sum() == fhv_zone["fhv_n"].sum())
tranches = ["fhv_n_nuit", "fhv_n_matin", "fhv_n_journee", "fhv_n_fin_aprem", "fhv_n_soiree"]
print("Somme des 5 tranches = total :", fhv_zone[tranches].sum(axis=1).sum() == fhv_zone["fhv_n"].sum())

cible = os.path.join(AGG_DIR, "fhv_zone.csv")
fhv_zone.to_csv(cible, index=False)
print("Sauvegardé :", cible)

Courses dans la table nettoyée : 1044984
Courses dans le résumé par zone : 1044984
2024 + 2025 = total : True
Somme des 5 tranches = total : False
Sauvegardé : c:\Users\DELL\Desktop\esprit\ML\notebook_v0\Nootebook_vf\agg_data\fhv_zone.csv


## préparation de la lecture de HVFHV

On refait les imports et on redéfinit les dossiers pour que la cellule fonctionne seule, puis on libère la mémoire de FHV (son résumé est déjà dans `agg_data/fhv_zone.csv`).

On ne charge pas tout le fichier : on lit seulement 5 lignes pour vérifier les 14 colonnes. Les 12 colonnes d'origine plus `duree_min` et `attente_min`, calculées au nettoyage. La lecture complète se fera par blocs à la cellule 14.

**Effet sur les données :** aucun, on vérifie seulement la structure du fichier.

In [10]:
import os, time, gc, warnings
import numpy as np
import pandas as pd
from IPython.display import display
warnings.filterwarnings("ignore")

CLEAN_DIR = os.path.join(os.getcwd(), "clean_data")
AGG_DIR = os.path.join(os.getcwd(), "agg_data")
os.makedirs(AGG_DIR, exist_ok=True)

if "fhv" in globals():
    del fhv
    gc.collect()

CHEMIN_HVFHV = os.path.join(CLEAN_DIR, "hvfhv_clean.csv.gz")
TYPES = {"zone_depart": "int16", "zone_arrivee": "float32", "distance": "float32", "prix": "float32",
         "pourboire": "float32", "frais_cbd": "float32", "paie_chauffeur": "float32",
         "duree_min": "float32", "attente_min": "float32", "partage": "category", "acces_pmr": "category"}
DATES = ["date_demande", "date_debut", "date_fin"]

apercu = pd.read_csv(CHEMIN_HVFHV, dtype=TYPES, parse_dates=DATES, nrows=5)
print("Colonnes :", apercu.shape[1])
print(list(apercu.columns))
display(apercu)

Colonnes : 14
['date_demande', 'date_debut', 'date_fin', 'zone_depart', 'zone_arrivee', 'distance', 'prix', 'pourboire', 'frais_cbd', 'partage', 'acces_pmr', 'paie_chauffeur', 'duree_min', 'attente_min']


,date_demande,date_debut,date_fin,zone_depart,zone_arrivee,distance,prix,pourboire,frais_cbd,partage,acces_pmr,paie_chauffeur,duree_min,attente_min
0,2024-02-01 00:05:14,2024-02-01 00:08:56,2024-02-01 00:26:03,179,226.0,2.520,15.180000,0.0,0.0,N,N,12.970000,17.116667,3.700000
1,2024-02-01 00:46:46,2024-02-01 00:48:52,2024-02-01 01:04:40,211,233.0,2.901,28.360001,0.0,0.0,N,N,21.129999,15.800000,2.100000
2,2024-02-01 00:14:30,2024-02-01 00:15:42,2024-02-01 00:27:40,74,167.0,3.400,17.000000,0.0,0.0,N,N,11.220000,11.966666,1.200000
3,2024-02-01 00:44:09,2024-02-01 00:47:13,2024-02-01 01:30:21,164,189.0,10.490,52.650002,0.0,0.0,N,N,38.110001,43.133335,3.066667
4,2024-02-01 00:08:48,2024-02-01 00:11:39,2024-02-01 00:16:26,14,14.0,1.500,8.890000,1.0,0.0,N,N,5.390000,4.783333,2.850000


## fonction qui résume un bloc de courses par zone

On écrit une fonction `preparer_bloc` qui reçoit un bloc de courses, crée les compteurs et les sommes, puis les additionne par zone de départ. Elle sera appliquée à chaque bloc à la cellule suivante.

**Colonnes communes avec les taxis**
- `n`, `n_2024`, `n_2025`, `n_weekend`, les 5 tranches horaires, `n_aeroport`
- `n_duree` / `s_duree`, `n_dist` / `s_dist`, `n_prix` / `s_prix`
- `s_prix_pm` / `s_dist_pm` : prix et distance des courses où les deux sont valides, pour le prix par mile
- `n_cbd` : courses de 2025 qui ont payé le péage CBD

**Ce qui change par rapport aux taxis**
- `s_pourboire` / `s_prix_pb` : ici, **toutes** les courses dont le prix et le pourboire sont valides comptent, car tout est payé dans l'application et le pourboire est toujours enregistré. Il n'y a pas de colonne de paiement ni de part d'espèces.

**Colonnes propres à HVFHV**
- `n_attente` / `s_attente` : nombre et somme des temps d'attente valides, pour l'attente moyenne
- `n_partage_ok` / `n_partage` : courses dont l'indicateur de partage est connu (Y ou N), et courses partagées (Y)
- `n_pmr_ok` / `n_pmr` : même chose pour les demandes de véhicule accessible aux fauteuils roulants
- `s_paie` / `s_prix_paie` : paie du chauffeur et prix des courses où les deux sont valides, pour la part du prix qui revient au chauffeur

**Effet sur les données :** aucun pour l'instant, la fonction est seulement définie.

In [12]:
AEROPORTS = [1, 132, 138]

def preparer_bloc(df):
    heure = df["date_debut"].dt.hour
    annee = df["date_debut"].dt.year

    ind = pd.DataFrame({"zone_id": df["zone_depart"]})
    ind["n"] = np.ones(len(df), dtype="int32")
    ind["n_2024"] = (annee == 2024).astype("int32")
    ind["n_2025"] = (annee == 2025).astype("int32")
    ind["n_weekend"] = (df["date_debut"].dt.dayofweek >= 5).astype("int32")
    ind["n_nuit"] = (heure < 6).astype("int32")
    ind["n_matin"] = ((heure >= 6) & (heure < 10)).astype("int32")
    ind["n_journee"] = ((heure >= 10) & (heure < 16)).astype("int32")
    ind["n_fin_aprem"] = ((heure >= 16) & (heure < 20)).astype("int32")
    ind["n_soiree"] = (heure >= 20).astype("int32")
    ind["n_aeroport"] = df["zone_arrivee"].isin(AEROPORTS).astype("int32")

    ind["n_duree"] = df["duree_min"].notna().astype("int32")
    ind["s_duree"] = df["duree_min"].fillna(0).astype("float64")
    ind["n_dist"] = df["distance"].notna().astype("int32")
    ind["s_dist"] = df["distance"].fillna(0).astype("float64")
    ind["n_prix"] = df["prix"].notna().astype("int32")
    ind["s_prix"] = df["prix"].fillna(0).astype("float64")

    prix_et_dist = df["prix"].notna() & df["distance"].notna()
    ind["s_prix_pm"] = df["prix"].where(prix_et_dist, 0).astype("float64")
    ind["s_dist_pm"] = df["distance"].where(prix_et_dist, 0).astype("float64")

    prix_et_pb = df["prix"].notna() & df["pourboire"].notna()
    ind["s_pourboire"] = df["pourboire"].where(prix_et_pb, 0).astype("float64")
    ind["s_prix_pb"] = df["prix"].where(prix_et_pb, 0).astype("float64")

    ind["n_cbd"] = ((annee == 2025) & (df["frais_cbd"] > 0)).astype("int32")

    ind["n_attente"] = df["attente_min"].notna().astype("int32")
    ind["s_attente"] = df["attente_min"].fillna(0).astype("float64")

    ind["n_partage_ok"] = df["partage"].isin(["Y", "N"]).astype("int32")
    ind["n_partage"] = (df["partage"] == "Y").astype("int32")
    ind["n_pmr_ok"] = df["acces_pmr"].isin(["Y", "N"]).astype("int32")
    ind["n_pmr"] = (df["acces_pmr"] == "Y").astype("int32")

    prix_et_paie = df["prix"].notna() & df["paie_chauffeur"].notna()
    ind["s_paie"] = df["paie_chauffeur"].where(prix_et_paie, 0).astype("float64")
    ind["s_prix_paie"] = df["prix"].where(prix_et_paie, 0).astype("float64")

    return ind.groupby("zone_id").sum()



##  lecture par blocs et passage de la course à la zone

On lit `hvfhv_clean.csv.gz` par blocs de 2 millions de courses. Chaque bloc est résumé par zone avec `preparer_bloc`, puis le bloc est oublié avant de lire le suivant. À la fin, on additionne tous les petits résumés zone par zone : comme ce sont des comptes et des sommes, le total est exactement le même que si on avait tout lu d'un coup.

Les colonnes reçoivent le préfixe `hvfhv_` et les sommes sont arrondies à 2 décimales.

**Effet sur les données :** on passe de dizaines de millions de courses à une ligne par zone et 30 colonnes (le numéro de zone et 29 totaux), sans jamais charger toute la table en mémoire. Cette cellule est la plus longue du notebook, car il faut décompresser et lire tout le fichier.

In [13]:
t0 = time.time()
resumes = []
nb_lues = 0

for i, bloc in enumerate(pd.read_csv(CHEMIN_HVFHV, dtype=TYPES, parse_dates=DATES, chunksize=2_000_000), start=1):
    nb_lues += len(bloc)
    resumes.append(preparer_bloc(bloc))
    print("\rbloc", i, "-", nb_lues, "courses traitées", end="")

hvfhv_zone = pd.concat(resumes).groupby(level=0).sum()
del resumes, bloc
gc.collect()

colonnes_s = [c for c in hvfhv_zone.columns if c.startswith("s_")]
hvfhv_zone[colonnes_s] = hvfhv_zone[colonnes_s].round(2)
hvfhv_zone.columns = ["hvfhv_" + c for c in hvfhv_zone.columns]
hvfhv_zone = hvfhv_zone.reset_index()

print("\nTemps (s) :", round(time.time() - t0))
print("Zones :", len(hvfhv_zone), "| colonnes :", hvfhv_zone.shape[1])
display(hvfhv_zone.sort_values("hvfhv_n", ascending=False).head(10))

bloc 3 - 4048744 courses traitées
Temps (s) : 26
Zones : 259 | colonnes : 30


,zone_id,hvfhv_n,hvfhv_n_2024,hvfhv_n_2025,hvfhv_n_weekend,hvfhv_n_nuit,hvfhv_n_matin,hvfhv_n_journee,hvfhv_n_fin_aprem,hvfhv_n_soiree,...,hvfhv_s_prix_pb,hvfhv_n_cbd,hvfhv_n_attente,hvfhv_s_attente,hvfhv_n_partage_ok,hvfhv_n_partage,hvfhv_n_pmr_ok,hvfhv_n_pmr,hvfhv_s_paie,hvfhv_s_prix_paie
134,138,73759,36330,37429,18235,4106,6113,24819,9454,20239,...,4002197.54,15867,73598,469346.78,73759,1,73759,56,2836426.30,4001859.54
128,132,69021,34514,34507,21389,8613,8440,17092,8082,19015,...,4699565.95,8914,68904,455878.68,69021,470,69021,240,3784339.61,4699416.18
59,61,53976,27647,26329,19248,7152,8730,13880,6382,11864,...,1116843.36,1661,53369,265498.23,53976,2654,53976,147,899398.87,1116370.87
77,79,53623,28522,25101,23806,12574,5259,9875,5458,15664,...,1401916.95,24930,53251,231395.03,53623,1962,53623,57,1008896.62,1401814.80
225,230,48740,24697,24043,14535,7014,5270,11347,5294,14669,...,1857508.22,23930,47792,231353.43,48740,716,48740,84,1251058.97,1857414.51
157,161,47412,23790,23622,9979,4272,3007,12438,7021,14112,...,1758373.05,23483,46972,216424.95,47412,587,47412,71,1141597.34,1758165.70
226,231,45744,23590,22154,14312,4833,6195,12228,5869,11220,...,1455134.30,21943,45411,184927.00,45744,686,45744,45,987999.90,1454956.95
74,76,44571,22464,22107,13318,4977,9440,12542,5065,7647,...,830675.25,295,44138,212785.47,44571,2369,44571,151,691395.19,830379.06
35,37,44201,22216,21985,18409,9587,5942,9638,4446,10309,...,968036.33,2230,43810,203401.38,44201,1965,44201,69,776434.65,967909.94
66,68,43509,22357,21152,13785,5344,5693,11668,5593,10291,...,1396815.71,21015,43028,179456.93,43509,935,43509,77,974475.69,1396727.12


## contrôle et sauvegarde

On vérifie que l'agrégation n'a ni perdu ni dupliqué de courses :
- le nombre de courses lues dans le fichier = somme de `hvfhv_n` sur toutes les zones ;
- courses 2024 + courses 2025 = total ;
- somme des 5 tranches horaires = total.

Puis on sauvegarde le résumé dans `agg_data/hvfhv_zone.csv`.

**Effet sur les données :** aucun changement, le résumé HVFHV par zone est gardé sur le disque. Les 4 résumés par zone sont maintenant prêts pour l'intégration.

In [15]:
print("Courses  :", nb_lues)
print("Courses par zone :", hvfhv_zone["hvfhv_n"].sum())
print("2024 + 2025 = total :", (hvfhv_zone["hvfhv_n_2024"] + hvfhv_zone["hvfhv_n_2025"]).sum() == hvfhv_zone["hvfhv_n"].sum())
tranches = ["hvfhv_n_nuit", "hvfhv_n_matin", "hvfhv_n_journee", "hvfhv_n_fin_aprem", "hvfhv_n_soiree"]


cible = os.path.join(AGG_DIR, "hvfhv_zone.csv")
hvfhv_zone.to_csv(cible, index=False)
print(":", cible)

Courses  : 4048744
Courses par zone : 4048744
2024 + 2025 = total : True
: c:\Users\DELL\Desktop\esprit\ML\notebook_v0\Nootebook_vf\agg_data\hvfhv_zone.csv
